## Imports

In [20]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql import functions as F
from pyspark.sql.dataframe import DataFrame as DF


spark = SparkSession.builder.master("local").appName("BigData").getOrCreate()


## Read the parquet files

In [21]:
df_products = spark.read.parquet("../data/1_bronze/products")
df_order_items = spark.read.parquet("../data/1_bronze/order_items")
df_translation = spark.read.parquet("../data/1_bronze/translation")

df_products.printSchema()     
df_order_items.printSchema()  
df_translation.printSchema()  

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)



## Cleaning

### Doubles


In [22]:
# Compter les occurrences par clé
df_counts_products = df_products.groupBy("product_id").count()
doublons_products = df_counts_products.filter(F.col("count") > 1)
doublons_products.show()

'''df_counts_order_items = df_order_items.groupBy("order_id").count()
doublons_order_items = df_counts_order_items.filter(F.col("count") > 1)
doublons_order_items.show()
print(doublons_order_items.count())'''

df_counts_translation = df_translation.groupBy("product_category_name").count()
doublons_translation = df_counts_translation.filter(F.col("count") > 1)
doublons_translation.show()

+----------+-----+
|product_id|count|
+----------+-----+
+----------+-----+

+---------------------+-----+
|product_category_name|count|
+---------------------+-----+
+---------------------+-----+



### Dates

In [23]:
df_order_items.select("shipping_limit_date").show(5, truncate=False)
df_order_items.printSchema()

+-------------------+
|shipping_limit_date|
+-------------------+
|2017-05-22 16:05:14|
|2017-03-15 14:09:17|
|2017-03-15 14:09:17|
|2017-03-15 14:09:17|
|2017-03-15 14:09:17|
+-------------------+
only showing top 5 rows
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)



### Null to false

In [24]:
#compter les valeurs nulles par colonne


df_products.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_products.columns]).show()
df_translation.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_translation.columns]).show()
df_order_items.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_order_items.columns]).show()

+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|         0|                  610|                610|                       610|               610|               2|                2|                2|               2|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+

+---------------------+-----------------------------+
|product_category_name|product_category_name_english|
+---------------------+-------------

### Export en parquet

In [25]:
#exporter les fichiers mis à jour dans le dossier 2_silver
df_products.write.mode("overwrite").parquet("../data/2_silver/products")
df_order_items.write.mode("overwrite").parquet("../data/2_silver/order_items")
df_translation.write.mode("overwrite").parquet("../data/2_silver/translation")